# 04.2 — Word2Vec Skip-gram

**Problem it solves:** Learn dense word representations that capture meaning, similarity, and analogy relationships — from raw text, no labels needed.

**Skip-gram:** Given a center word, predict its context words. This forces the model to put similar words (same contexts) close together in vector space.

**Negative sampling:** Computing softmax over 100k vocab is too slow. Instead, train a binary classifier: is this a real (word, context) pair or a random (word, noise) pair?

---

In [ ]:
import numpy as np
import re
from collections import Counter
import random

def tokenize(text): return re.findall(r'\b[a-z]+\b', text.lower())

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class Word2VecSkipGram:
    """
    Skip-gram with Negative Sampling (SGNS).
    
    Two embedding matrices:
    - W_in:  'center' word embeddings (what we use after training)
    - W_out: 'context' word embeddings (discarded after training)
    
    Objective: maximize log P(context|center) - log P(noise|center)
    """
    
    def __init__(self, embed_dim=50, window=2, n_negative=5, 
                 lr=0.025, n_epochs=5, min_count=1, subsample_t=1e-4):
        self.embed_dim = embed_dim
        self.window = window
        self.n_negative = n_negative
        self.lr = lr
        self.n_epochs = n_epochs
        self.min_count = min_count
        self.subsample_t = subsample_t
        self.vocab = []
        self.word2idx = {}
        self.W_in = None
        self.W_out = None
        self.noise_dist = None  # for negative sampling
    
    def _build_vocab(self, corpus: list[list[str]]):
        counts = Counter(w for sent in corpus for w in sent)
        self.vocab = [w for w, c in counts.most_common() if c >= self.min_count]
        self.word2idx = {w: i for i, w in enumerate(self.vocab)}
        
        # Noise distribution for negative sampling:
        # P(w)^0.75 (unigram distribution raised to 3/4 power)
        # This downsamples frequent words, giving rare words more negative samples
        freqs = np.array([counts[w] for w in self.vocab], dtype=float)
        freqs_power = freqs ** 0.75
        self.noise_dist = freqs_power / freqs_power.sum()
        
        return counts
    
    def _subsample(self, tokens: list[str], counts: Counter, total: int) -> list[str]:
        """
        Subsampling: frequent words like 'the' are downsampled.
        P(discard) = 1 - sqrt(t / f(w))  where t is subsample_t threshold.
        This is why Word2Vec works well: common words provide little information.
        """
        result = []
        for w in tokens:
            if w not in self.word2idx:
                continue
            freq = counts[w] / total
            keep_prob = min(1.0, (self.subsample_t / freq) ** 0.5 + self.subsample_t / freq)
            if random.random() < keep_prob:
                result.append(w)
        return result
    
    def _negative_sample(self, center_idx: int, k: int) -> list[int]:
        """Sample k negative words, avoiding the center word."""
        negs = []
        while len(negs) < k:
            idx = np.random.choice(len(self.vocab), p=self.noise_dist)
            if idx != center_idx:
                negs.append(idx)
        return negs
    
    def fit(self, corpus: list[str]):
        tokenized = [tokenize(sent) for sent in corpus]
        counts = self._build_vocab(tokenized)
        total_tokens = sum(counts.values())
        V = len(self.vocab)
        
        # Initialize embeddings: small random values
        # W_in[i] = embedding of word i as center
        # W_out[i] = embedding of word i as context
        self.W_in  = (np.random.rand(V, self.embed_dim) - 0.5) / self.embed_dim
        self.W_out = np.zeros((V, self.embed_dim))
        
        print(f"Vocab: {V} words")
        
        for epoch in range(self.n_epochs):
            total_loss = 0.0
            n_pairs = 0
            
            for sent in tokenized:
                tokens = self._subsample(sent, counts, total_tokens)
                
                for i, center_word in enumerate(tokens):
                    c_idx = self.word2idx[center_word]
                    
                    # Dynamic window size (like original paper)
                    win = random.randint(1, self.window)
                    
                    context_words = (
                        tokens[max(0, i-win):i] + 
                        tokens[i+1:i+win+1]
                    )
                    
                    for ctx_word in context_words:
                        ctx_idx = self.word2idx[ctx_word]
                        
                        # Positive pair
                        v_c = self.W_in[c_idx]    # center embedding
                        u_o = self.W_out[ctx_idx]  # context embedding
                        pos_score = np.dot(v_c, u_o)
                        pos_loss = -np.log(sigmoid(pos_score) + 1e-10)
                        
                        # Negative samples
                        neg_indices = self._negative_sample(c_idx, self.n_negative)
                        neg_loss = 0.0
                        
                        # Gradients
                        grad_v_c = (sigmoid(pos_score) - 1) * u_o  # grad for center from positive
                        grad_u_o = (sigmoid(pos_score) - 1) * v_c
                        
                        self.W_out[ctx_idx] -= self.lr * grad_u_o
                        
                        for neg_idx in neg_indices:
                            u_n = self.W_out[neg_idx]
                            neg_score = np.dot(v_c, u_n)
                            neg_loss += -np.log(sigmoid(-neg_score) + 1e-10)
                            
                            grad_neg = sigmoid(neg_score) * u_n
                            grad_v_c += grad_neg
                            
                            self.W_out[neg_idx] -= self.lr * sigmoid(neg_score) * v_c
                        
                        self.W_in[c_idx] -= self.lr * grad_v_c
                        total_loss += pos_loss + neg_loss
                        n_pairs += 1
            
            # Learning rate decay
            self.lr *= 0.9
            avg_loss = total_loss / max(n_pairs, 1)
            print(f"  Epoch {epoch+1}: avg_loss={avg_loss:.4f}  lr={self.lr:.5f}")
        
        return self
    
    def get_vector(self, word: str) -> np.ndarray:
        if word not in self.word2idx:
            return np.zeros(self.embed_dim)
        return self.W_in[self.word2idx[word]]
    
    def most_similar(self, word: str, n=5) -> list:
        if word not in self.word2idx:
            return []
        vec = self.get_vector(word)
        # Normalize
        norms = np.linalg.norm(self.W_in, axis=1, keepdims=True)
        norms[norms == 0] = 1
        normed = self.W_in / norms
        sims = normed @ (vec / (np.linalg.norm(vec) + 1e-10))
        top_idx = sims.argsort()[::-1][1:n+1]
        return [(self.vocab[i], float(sims[i])) for i in top_idx]
    
    def analogy(self, pos1, neg1, pos2) -> list:
        """king - man + woman ≈ queen"""
        v = (self.get_vector(pos1) - self.get_vector(neg1) + self.get_vector(pos2))
        v = v / (np.linalg.norm(v) + 1e-10)
        exclude = {pos1, neg1, pos2}
        norms = np.linalg.norm(self.W_in, axis=1, keepdims=True)
        norms[norms==0] = 1
        sims = (self.W_in / norms) @ v
        top = sims.argsort()[::-1][:20]
        return [(self.vocab[i], float(sims[i])) for i in top if self.vocab[i] not in exclude][:5]

# Train on larger corpus
corpus = [
    "the king rules the kingdom and the queen rules beside him",
    "the man walked to the market and the woman stayed home",
    "paris is the capital of france and london is the capital of england",
    "the dog is a loyal animal and the cat is an independent animal",
    "machine learning and deep learning are subfields of artificial intelligence",
    "neural networks learn representations from data automatically",
    "the cat sat on the mat and the dog sat on the rug",
    "king and queen rule the kingdom together as man and woman",
    "france is in europe and japan is in asia",
    "the computer processes data and the brain processes thoughts",
    "the boy played with toys and the girl read books",
    "a man is strong and a woman is also strong",
] * 20  # repeat to get more training signal

model = Word2VecSkipGram(embed_dim=30, window=3, n_negative=5, 
                          lr=0.05, n_epochs=10)
model.fit(corpus)

In [ ]:
# Inspect learned embeddings
print("Most similar words:")
for word in ['king', 'france', 'cat', 'learning']:
    if word in model.word2idx:
        similar = model.most_similar(word, n=5)
        print(f"\n  {word}: {similar}")

print("\nAnalogy (king - man + woman = ?):")
if all(w in model.word2idx for w in ['king', 'man', 'woman']):
    print(model.analogy('king', 'man', 'woman'))

## Summary

**Why negative sampling works:** Full softmax requires V multiplications per update. Negative sampling only requires k+1 (k negatives + 1 positive). At k=5 and V=100k, that's 20000x faster.

**What the model learns:** Word vectors where dot product approximates PMI. Similar contexts → similar vectors → geometry captures semantics.

**What breaks Word2Vec:**
- One vector per word — polysemy is ignored
- No morphological structure (GloVe/FastText improve this)
- Context window is local — can't capture long-range dependencies
- Training instability with very high/low learning rates

---
**Next:** 04.3 — Word2Vec CBOW